# ĐỒ ÁN CƠ SỞ: PHÁT HIỆN PHISHING URL BẰNG MACHINE LEARNING
## Bước 3: Phân tích dữ liệu (EDA) và Huấn luyện mô hình (Model Training)

**Mục tiêu của Notebook này:**
1. Khám phá tập dữ liệu 39 đặc trưng đã trích xuất.
2. So sánh hiệu suất của 6 thuật toán phổ biến.
3. Tối ưu hóa siêu tham số cho XGBoost.
4. Xuất mô hình để đưa vào Chrome Extension.

In [ ]:
# CELL 1: IMPORT THƯ VIỆN
import os
import json
import time
import numpy as np
import pandas as pd  # Thư viện xử lý dữ liệu dạng bảng
import matplotlib.pyplot as plt  # Thư viện vẽ đồ thị cơ bản
import seaborn as sns  # Thư viện vẽ biểu đồ thống kê đẹp hơn

# Các công cụ chia dữ liệu và tính toán chỉ số đo lường
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import (
    accuracy_score, f1_score, roc_auc_score, confusion_matrix,
    classification_report, roc_curve, precision_recall_curve
)
from sklearn.preprocessing import StandardScaler

# Danh sách các thuật toán Machine Learning sử dụng để đối chứng
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
import lightgbm as lgb
import xgboost as xgb

In [ ]:
# CELL 2: NẠP DỮ LIỆU TỪ FILE CSV
DATA_DIR = 'data'
# Ưu tiên nạp file v4 (39 features), nếu không thấy thì nạp file mặc định
file_path = os.path.join(DATA_DIR, 'features_v4.csv') if os.path.exists(os.path.join(DATA_DIR, 'features_v4.csv')) else os.path.join(DATA_DIR, 'features.csv')
df = pd.read_csv(file_path)

# Kiểm tra tổng quan bộ dữ liệu
print(f"Tệp đang dùng: {file_path}")
print(f"Kích thước: {df.shape[0]:,} dòng URL x {df.shape[1]} cột")

# Đếm số lượng mẫu của từng lớp để kiểm tra tính cân bằng
phish_count = df['label'].sum()
legit_count = (df['label'] == 0).sum()
print(f"Phishing (1): {phish_count:,} mẫu ({phish_count/len(df)*100:.1f}%)")
print(f"An toàn   (0): {legit_count:,} mẫu ({legit_count/len(df)*100:.1f}%)")

In [ ]:
# CELL 3: PHÂN TÍCH TƯƠNG QUAN (EDA)

# Vẽ biểu đồ cột so sánh số lượng 2 lớp
plt.figure(figsize=(6, 4))
df['label'].value_counts().plot(kind='bar', color=['#4CAF50', '#f44336'])
plt.title('So sánh số lượng mẫu: An toàn vs Lừa đảo')
plt.ylabel('Số lượng')
plt.xticks([0, 1], ['An toàn (0)', 'Lừa đảo (1)'], rotation=0)
plt.show()

# Vẽ bản đồ nhiệt (Heatmap) để xem các đặc trưng nào có quan hệ mạnh với nhãn 'label'
plt.figure(figsize=(12, 10))
correlation = df.corr()  # Tính ma trận tương quan
sns.heatmap(correlation, cmap='coolwarm', center=0)
plt.title('Bản đồ tương quan (Heatmap) giữa 39 đặc trưng')
plt.show()

In [ ]:
# CELL 4: CHIA DỮ LIỆU ĐỂ HỌC VÀ KIỂM TRA
X = df.drop('label', axis=1)  # Đầu vào là 39 đặc trưng số
y = df['label']               # Đầu ra là nhãn (1 hoặc 0)

# Chia theo tỷ lệ 80% để Train (dạy máy) và 20% để Test (chấm điểm)
# random_state=42 để kết quả luôn giống nhau mỗi khi chạy lại
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Dữ liệu dạy máy (Train): {len(X_train):,} mẫu")
print(f"Dữ liệu chấm thi (Test):  {len(X_test):,} mẫu")

In [ ]:
# CELL 5: THIẾT LẬP 6 THUẬT TOÁN ĐỐI CHỨNG
models = {
    # Mô hình tuyến tính cơ bản, max_iter=1000 để đảm bảo thuật toán hội tụ (tìm ra đáp án)
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),

    # Rừng ngẫu nhiên: sử dụng 100 cây quyết định để biểu quyết kết quả
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42),

    # Thuật toán mạnh nhất hiện tại: học từ sai số của các cây trước đó
    'XGBoost': xgb.XGBClassifier(
        n_estimators=100,    # Số lượng cây huấn luyện ban đầu
        max_depth=5,         # Độ sâu tối đa của mỗi cây (tránh quá phức tạp)
        learning_rate=0.1,   # Tốc độ học (0.1 là mức cân bằng giữa nhanh và chính xác)
        random_state=42, 
        eval_metric='logloss'# Hàm đo lường mất mát dữ liệu
    ),

    # Tương tự XGBoost nhưng tối ưu tốc độ và bộ nhớ hơn
    'LightGBM': lgb.LGBMClassifier(
        n_estimators=100, 
        max_depth=5, 
        learning_rate=0.1, 
        random_state=42
    ),

    # Cây quyết định đơn lẻ: dễ hiểu nhưng dễ bị học vẹt, giới hạn độ sâu 10 để kiểm soát
    'Decision Tree': DecisionTreeClassifier(max_depth=10, random_state=42),

    # Tìm siêu phẳng phân tách: rất mạnh nhưng chạy chậm trên dữ liệu lớn
    'SVM': SVC(probability=True, random_state=42)  # probability=True để lấy được xác suất % dự đoán
}

In [ ]:
# CELL 6: VÒNG LẶP HUẤN LUYỆN VÀ ĐÁNH GIÁ
results = []

print(f"{'Thuật toán':25s} {'Chính xác':>10s} {'F1-Score':>8s} {'Bắt nhầm(FPR)':>10s}")
print("-"*60)

for name, model in models.items():
    start = time.time()
    model.fit(X_train, y_train)  # Cho máy học
    
    y_pred = model.predict(X_test)  # Dự đoán thử trên tập Test
    y_prob = model.predict_proba(X_test)[:, 1] # Lấy xác suất % để vẽ biểu đồ
    
    # Tính toán các chỉ số
    acc = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    
    # Tính FPR (Tỷ lệ báo động giả - cực kỳ quan trọng)
    cm = confusion_matrix(y_test, y_pred)
    tn, fp, fn, tp = cm.ravel() 
    fpr = (fp / (fp + tn)) * 100
    
    train_time = time.time() - start
    results.append({'model': name, 'accuracy': acc, 'f1': f1, 'fpr': fpr, 'time': train_time, 'model_obj': model})
    
    print(f"{name:25s} {acc*100:>9.2f}% {f1:>8.4f} {fpr:>9.2f}%")

print("-"*60)

In [ ]:
# CELL 8: XEM ĐẶC TRƯNG NÀO QUAN TRỌNG NHẤT (XGBoost)
xgb_model = [r['model_obj'] for r in results if r['model'] == 'XGBoost'][0]
importances = xgb_model.feature_importances_  # Lấy trọng số đóng góp của từng cột

# Lấy Top 15 đặc trưng ảnh hưởng nhất đến kết luận lừa đảo
indices = np.argsort(importances)[-15:]

plt.figure(figsize=(10, 8))
plt.barh(range(15), importances[indices], color='orange')
plt.yticks(range(15), [X.columns[i] for i in indices])
plt.title('Top 15 đặc trưng quan trọng nhất trong mô hình XGBoost')
plt.xlabel('Mức độ đóng góp (Importance)')
plt.show()

In [ ]:
# CELL 9: TỐI ƯU HÓA SIÊU THAM SỐ (HYPERPARAMETER TUNING)
from sklearn.model_selection import RandomizedSearchCV

# Không gian tìm kiếm các nút vặn tốt nhất cho XGBoost
param_dist = {
    'n_estimators': [100, 200, 300], # Số lượng cây
    'max_depth': [3, 5, 7],          # Độ sâu của cây
    'learning_rate': [0.01, 0.1],    # Tốc độ học
    'reg_alpha': [0, 1, 5],          # Hệ số phạt lỗi (L1 regularization)
    'gamma': [0, 0.3, 0.5],          # Ngưỡng tối thiểu để tách nhánh
}

print("\nĐang tìm bộ tham số tối ưu (Tuning) giúp mô hình không bị 'học vẹt'...")
xgb_base = xgb.XGBClassifier(random_state=42, eval_metric='logloss')

# Thử nghiệm ngẫu nhiên 20 tổ hợp khác nhau, chấm điểm bằng F1-Score
random_search = RandomizedSearchCV(xgb_base, param_distributions=param_dist, n_iter=20, cv=3, scoring='f1', random_state=42, n_jobs=-1)
random_search.fit(X_train, y_train)

print(f"✓ Kết quả tốt nhất: {random_search.best_params_}")
print(f"✓ Điểm F1 tương ứng: {random_search.best_score_:.4f}")